# VibeVoice voice-cloning TTS on Colab

Runs VibeVoice-1.5B through Hugging Face Transformers (native support since 5.17) on a free **T4 GPU**.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells top to bottom.
2. Part A is a smoke test with a public sample voice. Listen to it before going on.
3. Part B is a bake-off: upload a bundle zip with reference voices and a `lines.json`, and it generates every line with several seeds and scores each take's length against a target window.
4. The last cell zips everything for download.

Weights: `vibevoice/VibeVoice-1.5B-hf`, pinned to a revision. MIT licence. Microsoft recommends research use and forbids cloning a real person's voice without recorded consent.

In [ ]:
# 1. Check the GPU
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1e9, 1), "GB")

In [ ]:
# 2. Install pinned packages (about a minute). Runs before transformers is imported, so no restart is needed.
!pip install -q -r https://raw.githubusercontent.com/tbalt88/VibeVoice/kaitona/kaitona/requirements.txt
import transformers, diffusers
print("transformers", transformers.__version__, "| diffusers", diffusers.__version__)

In [ ]:
# 3. Load the model (downloads 5.4 GB on first run, a few minutes)
import os, json, time
import numpy as np, soundfile as sf
from IPython.display import Audio, display
from transformers import AutoProcessor, AutoModelForTextToWaveform, set_seed

MODEL_ID = "vibevoice/VibeVoice-1.5B-hf"
REVISION = "d4cd82eb4371bf3e3d088e1ea7fa4eefb60449bf"
# T4 has no native bfloat16. Start with float16; if takes come out silent or as noise,
# switch to torch.float32 and rerun this cell.
DTYPE = torch.float16

processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForTextToWaveform.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=DTYPE, device_map="cuda", attn_implementation="sdpa")
model.eval()
SR = processor.feature_extractor.sampling_rate
OUT = "/content/out"
os.makedirs(OUT, exist_ok=True)
print("loaded", MODEL_ID, "|", DTYPE, "|", SR, "Hz |", round(torch.cuda.memory_allocated() / 1e9, 1), "GB on GPU")

In [ ]:
# 4. Helpers
def tts(turns, out_path, seed=42, max_new_tokens=150, diffusion_steps=10):
    """turns: list of (role, text, ref). ref is a path, URL or None; give it on a role's first turn to clone that voice."""
    conv = []
    for role, text, ref in turns:
        content = [{"type": "text", "text": text}]
        if ref:
            content.append({"type": "audio", ("url" if ref.startswith("http") else "path"): ref})
        conv.append({"role": str(role), "content": content})
    set_seed(seed)
    inputs = processor.apply_chat_template(conv, return_dict=True, tokenize=True, add_generation_prompt=True)
    inputs = inputs.to(model.device, model.dtype)
    t0 = time.time()
    with torch.inference_mode():
        audio = model.generate(**inputs, max_new_tokens=max_new_tokens, num_diffusion_steps=diffusion_steps)
    processor.save_audio(audio, out_path)
    return round(time.time() - t0, 1)


def voiced_len(x, sr):
    """Seconds from first to last voiced 10 ms frame. Same rule as kai-tona's fit_takes.py."""
    h, w = sr // 100, 3 * (sr // 100)
    e = np.array([20 * np.log10(np.sqrt((x[i:i + w] ** 2).mean()) + 1e-9) for i in range(0, len(x) - w, h)])
    if len(e) == 0:
        return 0.0
    on = np.where(e > max(min(np.percentile(e, 30) + 10, e.max() - 15), -45))[0]
    return round(float((on[-1] + 3 - on[0]) * 0.01), 2) if len(on) else 0.0


def measure(path):
    x, sr = sf.read(path, dtype="float32")
    x = x.mean(axis=1) if x.ndim > 1 else x
    ok = bool(np.isfinite(x).all()) and float(np.abs(x).max()) > 1e-3
    return {"dur_s": round(len(x) / sr, 2), "voiced_s": voiced_len(x, sr),
            "peak": round(float(np.abs(x).max()), 3), "sr": sr, "ok": ok}

## Part A: smoke test

Listen for three things: a clean voice, **no** spoken "generated by AI" disclaimer, and no background music. If `ok` is False, switch `DTYPE` to `torch.float32` in cell 3.

In [ ]:
# 5. Smoke test with a public sample voice
ALICE = "https://huggingface.co/datasets/bezzam/vibevoice_samples/resolve/main/voices/en-Alice_woman.wav"
p = f"{OUT}/smoke_test.wav"
secs = tts([("0", "Hello, this is a quick test of VibeVoice running on a free Colab GPU.", ALICE)], p, seed=42)
print("generated in", secs, "s |", measure(p))
display(Audio(p))

## Part B: bake-off

Upload a zip containing `lines.json` and the reference WAVs it names. Format:

```json
{"voices": {"kai": {"full": "kai_full.wav", "10s": "kai_10s.wav"}},
 "lines": [{"id": "S3-1a", "speaker": "kai", "text": "Take it.", "mouth_window_s": 0.29}]}
```

Each take is scored by voiced length against `mouth_window_s`: within 10% either way is `FIT`.

In [ ]:
# 6. Upload the bundle zip
import zipfile
from google.colab import files
up = files.upload()
name = next(iter(up))
BUNDLE = "/content/bundle"
zipfile.ZipFile(f"/content/{name}").extractall(BUNDLE)
spec = json.load(open(f"{BUNDLE}/lines.json", encoding="utf-8"))
print(len(spec["lines"]), "lines |", {k: list(v) for k, v in spec["voices"].items()})

In [ ]:
# 7. Generate every line x every reference x every seed (roughly 15-25 min on a T4)
SEEDS = [42, 7, 11]
report = []
os.makedirs(f"{OUT}/lines", exist_ok=True)
for ln in spec["lines"]:
    for ref_name, ref_file in spec["voices"][ln["speaker"]].items():
        for seed in SEEDS:
            p = f"{OUT}/lines/{ln['id']}_{ln['speaker']}_{ref_name}_s{seed}.wav"
            secs = tts([("0", ln["text"], f"{BUNDLE}/{ref_file}")], p, seed=seed)
            r = measure(p)
            fit = round(r["voiced_s"] / ln["mouth_window_s"], 2) if ln["mouth_window_s"] else None
            verdict = "FIT" if fit and 0.9 <= fit <= 1.1 else ("fast" if fit and fit < 0.9 else "slow")
            row = {**ln, "ref": ref_name, "seed": seed, "file": os.path.basename(p), "gen_s": secs,
                   **r, "fit": fit, "verdict": verdict}
            report.append(row)
            print(f"{ln['id']:6} {ln['speaker']:4} {ref_name:4} s{seed:<3} voiced {r['voiced_s']:5.2f}s"
                  f" / {ln['mouth_window_s']:.2f}s  x{fit}  {verdict:4}  {'ok' if r['ok'] else 'BAD AUDIO'}")
json.dump(report, open(f"{OUT}/report.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

In [ ]:
# 8. Summary: best take per line, closest to its window
best = {}
for r in report:
    if r["ok"] and (r["id"] not in best or abs(r["fit"] - 1) < abs(best[r["id"]]["fit"] - 1)):
        best[r["id"]] = r
for k, r in best.items():
    print(f"{k:6} {r['speaker']:4} {r['ref']:4} s{r['seed']:<3} x{r['fit']:<5} {r['verdict']:4} {r['file']}")
print(sum(r["verdict"] == "FIT" for r in best.values()), "of", len(best), "lines have a take within 10%")
for ref_name in sorted({r["ref"] for r in report}):
    rows = [r for r in report if r["ref"] == ref_name and r["ok"]]
    if rows:
        print(f"ref {ref_name}: median fit x{np.median([r['fit'] for r in rows]):.2f} over {len(rows)} takes")

In [ ]:
# 9. Optional: the whole scene in one pass, so the model handles turn-taking itself
for ref_name in ["10s", "full"]:
    seen, turns = set(), []
    first = spec["lines"][0]["speaker"]
    for ln in spec["lines"]:
        role = "0" if ln["speaker"] == first else "1"
        ref = None if ln["speaker"] in seen else f"{BUNDLE}/{spec['voices'][ln['speaker']][ref_name]}"
        seen.add(ln["speaker"])
        turns.append((role, ln["text"], ref))
    p = f"{OUT}/full_scene_{ref_name}.wav"
    secs = tts(turns, p, seed=42, max_new_tokens=700)
    print(ref_name, "generated in", secs, "s |", measure(p))
    display(Audio(p))

In [ ]:
# 10. Download everything
import shutil
shutil.make_archive("/content/vibevoice_out", "zip", OUT)
files.download("/content/vibevoice_out.zip")